# Notebook 08 (optional) — Analyst-recommendations basket (Track B: Free-only)

*Portfolio Intelligence Engine — User Guide Series, **Track B (free sources only)**, optional standalone chapter.*
[Series README](../portfolio/README.md) · [Story Bible](../portfolio/STORY_BIBLE.md) · Epic [#1428](https://github.com/prajoria/OpenBB/issues/1428) · This notebook [#1444](https://github.com/prajoria/OpenBB/issues/1444) · Track A counterpart: [`../portfolio/08-analyst-recommendations-basket.ipynb`](../portfolio/08-analyst-recommendations-basket.ipynb).

---

## Where this notebook fits (Track B)

The main 7-notebook Track B series (NB01→NB07) teaches how to *analyze* a basket under free-only providers. It never teaches how to *construct one from scratch*. This standalone chapter answers the two questions readers always ask after NB07 — what a defensible blank-page basket looks like, and which analyst sources the pros actually use — using ONLY free-authoritative sources (SEC + CBOE + yfinance-recorded).

By the end we can answer:

> *If I built a 16-ETF all-weather basket using published discipline > (Dalio, Bogle, Faber, Fidelity), what would the review numbers look > like under free-only providers, and how does it compare to the > 5-ETF low-cost self-maintained alternative?*

**Not a stock-picking recommendation.** Everything here is educational — no forward-return targets, no "buy this." Illustrative weights only.


## 0. Before we run anything

Same venv rule as every notebook in this series — `.venv_portfolio`. State goes into `.notebook_state/` (gitignored). We write to `analyst_basket_free.json` and `analyst_basket_low_cost_free.json` — do NOT overwrite the Track A files (`analyst_basket.json`, `analyst_basket_low_cost.json`).


In [ ]:
# [Track B / NB08 §0] environment sanity — assert .venv_portfolio + STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio, not the current interpreter.\n"
    "See NB01 §0 for setup.\n"
    f"Currently running: {sys.executable}"
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


## 0.5 Why free-only NB08 looks different from Track A

The basket-construction narrative (Dalio / Bogle / Faber / Fidelity / Vanguard) is framework-only text — it does not depend on any provider, so it lands unchanged. Three places where Track B honestly diverges:

1. **§5 look-through** — same N-PORT path Track A uses (post-PR #1445    Track A NB08 §5 already routes through SEC N-PORT). Track B reuses    the shared [`_nport_lookthrough.py`](../portfolio/_nport_lookthrough.py)    helper via `sys.path.insert`.
2. **§6 sector rollup** — Track A resolves the constituent-ticker tail    via `obb.equity.profile(provider="fmp_cached")`. Track B can't, so    it uses the same `FREE_SECTOR_MAP` + `Unknown` bucket discipline    NB03 (Track B) §4 shipped. Top ~60 mega-caps carry >80% of the    x-ray weight; the tail lands in `Unknown` and is named honestly.
3. **§7 MSFT spot-check** — the Analysis 7-phase composite requires    `fmp_cached`. Track B can't run it — we point Sam at Track A NB02 /    Track A NB08 §7 for the composite (`action_label='Avoid'`,    `composite_score≈2.51` on the last shipped run). Same honest hand-   off NB02 (Track B) §7 documented.

The 16-ETF Variant A and 5-ETF Variant B baskets, weights, backtest window, and framework prose are unchanged from Track A. Bare-term pointers (risk parity, all-weather, three-fund, sector rotation, target-date, trend-following, analyst price target, Zacks rank, Morningstar rating, SEC EDGAR, commodity ETF, rebalancing, efficient frontier, CAGR, Sharpe ratio, fundamental analysis, expense ratio) were all cited with Investopedia links in Track A NB08 — not re-cited here.


### Provider chain for this notebook (Track B / #1444)

| Data path | Provider (Track B) |
|---|---|
| ETF constituent look-through (all 16 ETFs) | SEC N-PORT (`sec`) via `_nport_lookthrough` |
| Prices / OHLCV for backtest | CBOE EOD (`cboe`) |
| Sector classification per constituent | `FREE_SECTOR_MAP` + `Unknown` bucket |
| MSFT spot-check (7-phase composite) | *hand-off to Track A NB02 / NB08 §7* |
| Options / IV context (if referenced) | `cboe` `OptionsChains` |
| Analyst grades / price targets | *no free authoritative source — gap documented in §11* |

**Non-N-PORT filers**: GLD (commodity grantor trust) and DBC (commodity pool) raise `NportUnavailable` from the helper and pass through as opaque single-symbol positions. Same behaviour as Track A.


## 1. Why this notebook + reader contract

Same story as Track A NB08 §1, retold under free-only providers.

NB03 (Track B) taught me how to x-ray a basket using SEC N-PORT. NB05 (Track B) taught me to attribute returns. NB06 (Track B) taught me to backtest under `provider='cboe'`. None of them taught me how to sit down with 16 empty slots and fill them with a defensible process rather than my last three CNBC-driven hunches — and how to do it without paying for a data provider.

The trader's problem: when I first tried to build a "proper" portfolio, I couldn't tell whether my weights came from a framework or from vibes. The frameworks exist — Dalio, Bogle, Faber, Fidelity all published their playbooks — but nobody hands the working retail trader a synthesized "here's how they compose." So I did the synthesis myself in §2, cited every framework line-by-line, and then ran the resulting basket through the same review NB03-NB06 (Track B) would run on any other book.

**What you get out of this Track B notebook:**

- A 16-ETF basket with each position's weight tied to a specific   published framework.
- A look-through view via SEC N-PORT that decomposes it into hundreds   of underlying issuers.
- HHI + effective-N (pure arithmetic — same numbers as Track A).
- A backtest over 2023-01-03 → 2024-12-31 vs SPY via   `provider='cboe'`.
- A 5-ETF Variant B low-cost self-maintained alternative + head-to-  head comparison.
- One code cell showing where to plug in your own "Fortress" basket.
- Curated analyst-research destinations (with the free-only gap   called out).

**What you do NOT get:** live scraping of analyst sites, a paid `fmp_cached` sector rollup, the Analysis 7-phase composite (requires `fmp_cached`), an intraday paper mark, or a claim that this basket will make money. That is beyond scope by design (see §11).


## 2. Portfolio construction — the 5 frameworks

Same five frameworks as Track A NB08 §2. Every seat at the retail table borrows from one of them; naming them explicitly lets you tell your framework apart from your hunches.

1. **Ray Dalio — All Weather / risk parity.** Four-quadrant macro    framework (growth up/down × inflation up/down) with a    30/40/15/7.5/7.5 target across stocks, long bonds, intermediate    bonds, gold, and commodities.
2. **Bogleheads — three-fund portfolio.** Total US + total intl +    total bond. Rebalance annually, ER < 10 bps. The reference bar    for any active retail strategy.
3. **Fidelity — sector rotation.** Business-cycle framework: early    = discretionary/financials; mid = tech/industrials; late =    energy/materials; recession = staples/utilities/health care.
4. **Vanguard — target-date fund.** Age-anchored glide-path; used    only as the mid-career reference (60/40-ish equity/bond).
5. **Meb Faber — Ivy Portfolio + trend-following overlay.** Equal-   weight across five asset classes; long-only when above the    10-month MA. Overlay not implemented here.

Full Investopedia glossary boxes are in Track A NB08 §2 — not re-cited. The 16-position basket in §4 is a **synthesis** — no one framework dominates. Broad market spine + fixed-income ballast is Bogleheads; the gold/long-bond/commodity kickers are Dalio; the sector-satellite tilts are Fidelity; and the equal-weight approach across asset classes is Faber.


## 3. Trader / analyst sources — where the pros look (and the free-tier gap)

Track A NB08 §3 lists TipRanks / Zacks / Morningstar / Seeking Alpha / ETF.com / SEC EDGAR with what each aggregates. The list is unchanged for Track B — those sites don't care whether you pay for FMP. What IS different for Track B: **published analyst opinion** (price targets, ratings, moat scores, sentiment) has NO free-authoritative API path. Track A pulls consensus estimates via `fmp_cached`; Track B can't. yfinance surfaces `recommendations` but it's a scraped best-effort feed with no SLA and questionable redistribution posture (see #1425).

Honest options for Track B readers:

1. **Click through to the aggregator sites manually** — TipRanks /    Zacks / Morningstar all show consensus targets on the free web    page. Fine for a single-name spot-check, useless for basket-   scale automation.
2. **Read the 13F filings directly** — SEC EDGAR is free-   authoritative for who owns what. Bridgewater, Renaissance,    Berkshire, ARK all file 13Fs; 45-day lag after quarter-end.    `openbb-sec` covers the query. NB04 (Track B) already uses this    path for smart-money conviction.
3. **Accept the gap and move on** — the framework-driven baskets    below don't need analyst grades to be defensible. Dalio / Bogle /    Faber / Fidelity all published their allocations without any    analyst overlay.

Six destinations worth bookmarking — same table as Track A NB08 §3, not re-inlined here to avoid drift. See [`../portfolio/08-analyst-recommendations-basket.ipynb`](../portfolio/08-analyst-recommendations-basket.ipynb) §3.


## 4. The 16-ETF Variant A basket — table + rationale

**Variant A — Diversified all-weather (16 ETFs, higher maintenance).**

Same basket as Track A NB08 §4, unchanged (frameworks don't care about your provider tier). Weights sum to 100%.

**Broad market spine (55%)** — Bogleheads three-fund plus Faber's REIT + Dalio's gold sleeve.

**Fixed-income ballast (25%)** — Bogleheads total-bond plus Dalio's long / short / TIPS split.

**Sector satellites (20%)** — Fidelity sector-rotation tilts that overweight the areas VTI is structurally light in (defensives + inflation-sensitive + EM diversifier).

*The code cell below writes the basket to `.notebook_state/analyst_basket_free.json` (Track B suffix — Track A's `analyst_basket.json` is left untouched) and prints the table.*


In [ ]:
# [Track B / NB08 §4] Build the 16-ETF Variant A basket + write JSON (with _free suffix)
import json
from pathlib import Path

BASKET = [
    # Broad market spine (55%)
    {"symbol": "VTI",  "weight": 0.30, "role": "US equity broad",         "framework": "Bogleheads three-fund"},
    {"symbol": "VXUS", "weight": 0.15, "role": "Ex-US developed + EM",    "framework": "Bogleheads three-fund"},
    {"symbol": "VNQ",  "weight": 0.05, "role": "REITs",                   "framework": "Faber Ivy"},
    {"symbol": "GLD",  "weight": 0.05, "role": "Inflation hedge (gold)",  "framework": "Dalio all-weather"},
    # Fixed-income ballast (25%)
    {"symbol": "BND",  "weight": 0.15, "role": "Investment-grade agg",    "framework": "Bogleheads three-fund"},
    {"symbol": "TLT",  "weight": 0.05, "role": "Long-duration hedge",     "framework": "Dalio all-weather"},
    {"symbol": "SHY",  "weight": 0.03, "role": "Short-duration liquidity", "framework": "Dalio all-weather"},
    {"symbol": "TIP",  "weight": 0.02, "role": "Real-rate hedge (TIPS)",  "framework": "Dalio all-weather"},
    # Sector satellites (20%)
    {"symbol": "XLE",  "weight": 0.03, "role": "Energy sector",           "framework": "Fidelity rotation"},
    {"symbol": "XLF",  "weight": 0.03, "role": "Financials sector",       "framework": "Fidelity rotation"},
    {"symbol": "XLV",  "weight": 0.03, "role": "Health care sector",      "framework": "Fidelity rotation (defensive)"},
    {"symbol": "XLU",  "weight": 0.02, "role": "Utilities sector",        "framework": "Fidelity rotation (defensive)"},
    {"symbol": "XLB",  "weight": 0.02, "role": "Materials sector",        "framework": "Fidelity rotation"},
    {"symbol": "XLI",  "weight": 0.02, "role": "Industrials sector",      "framework": "Fidelity rotation"},
    {"symbol": "DBC",  "weight": 0.03, "role": "Broad commodities",       "framework": "Dalio all-weather + Faber Ivy"},
    {"symbol": "VWO",  "weight": 0.02, "role": "Emerging markets equity", "framework": "Faber Ivy diversifier"},
]

total_w = sum(p["weight"] for p in BASKET)
assert abs(total_w - 1.0) < 1e-9, f"weights sum to {total_w}, not 1.0"

state = Path(".notebook_state")
state.mkdir(exist_ok=True)
# Track B path — DO NOT overwrite Track A's analyst_basket.json
basket_path = state / "analyst_basket_free.json"
basket_path.write_text(json.dumps(BASKET, indent=2), encoding="utf-8")

print(f"Wrote {basket_path}  ({len(BASKET)} ETFs, weights sum to {total_w*100:.1f}%)")
print("(Track A's .notebook_state/analyst_basket.json NOT modified by this notebook.)")
print()
print(f"{'Ticker':<6}{'Weight':>8}   {'Role':<32}Framework")
print("-" * 92)
for p in BASKET:
    print(f"{p['symbol']:<6}{p['weight']*100:>7.1f}%   {p['role']:<32}{p['framework']}")

# Convenience alias used in later cells
basket = BASKET


## 5. Basket X-Ray — SEC N-PORT look-through

Same discipline as NB03 (Track B) — SEC Form N-PORT is *the* filing every '40-Act US-registered fund files, so it's free-authoritative for both tracks. We reuse the shared [`_nport_lookthrough.py`](../portfolio/_nport_lookthrough.py) helper via `sys.path.insert(0, '../portfolio')`.

Equity ETFs (VTI, VXUS, VNQ, VWO, sector XL*) unwrap to their top issuers. Bond ETFs (BND, TLT, SHY, TIP) also file N-PORT and unwrap the same way. Commodity grantor trusts (GLD, DBC) raise `NportUnavailable` and pass through as opaque single-symbol lines — identical behaviour to Track A NB08 §5.

Every N-PORT fetch disk-caches under `.notebook_state/nport_cache/` so a kernel-restart re-run does not re-hit SEC EDGAR.


In [ ]:
# [Track B / NB08 §5] Look-through via SEC N-PORT (shared helper)
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("../portfolio").resolve()))
from _nport_lookthrough import effective_positions, NportUnavailable  # noqa: E402

EQUITY_ETFS = {
    "VTI", "VXUS", "VNQ", "VWO",
    "XLE", "XLF", "XLV", "XLU", "XLB", "XLI",
    "QQQ", "SPY", "DIA", "IWM", "VOO", "VEA",
}
BOND_ETFS = {"BND", "TLT", "SHY", "TIP", "SCHP", "AGG"}
COMMODITY_TRUSTS = {"GLD", "DBC", "SLV"}

effective, non_nport, opaque = effective_positions(
    basket,
    equity_etfs=EQUITY_ETFS,
    bond_etfs=BOND_ETFS,
    commodity_trusts=COMMODITY_TRUSTS,
    top_n_per_etf=50,  # top 50 issuers per ETF; tail bucketed as TAIL_<ETF>
)

print(f"Basket: {len(basket)} ETFs")
print(f"After N-PORT look-through: {len(effective)} distinct effective positions")
print(f"Total effective weight: {sum(effective.values())*100:.1f}%")
print()

if opaque:
    print("Opaque (no look-through possible):")
    for sym, reason in opaque:
        print(f"  {sym}: {reason[:80]}")
    print()

print("Top 15 effective positions:")
print(f"{'Issuer / bucket':<40}{'Weight':>10}")
print("-" * 52)
for key, w in sorted(effective.items(), key=lambda kv: -kv[1])[:15]:
    print(f"{str(key)[:38]:<40}{w*100:>9.2f}%")


## 6. Risk metrics — HHI + effective-N + free-tier sector view

Two concentration numbers to quote from now on:

- **HHI** — sum of squared weights, 1/N to 1. Pure arithmetic; the   number is identical to Track A NB08 §6 because it depends only on   the effective-position weights the N-PORT helper produced.
- **Effective-N** — 1 / HHI.

Naive vs look-through **sector** view is where Track B diverges. Track A rolls tickers up via `obb.equity.profile(provider=
"fmp_cached")`. Track B uses the same `FREE_SECTOR_MAP` + `Unknown` bucket pattern NB03 (Track B) §4 shipped. Top ~60 mega-caps carry >80% of the x-ray weight; the tail lands in `Unknown` — named honestly rather than fake-classified.


In [ ]:
# [Track B / NB08 §6] HHI + effective-N + sector view (free-tier)
# Sector rollup uses hand-curated FREE_SECTOR_MAP + Unknown bucket.
# NO paid-provider calls in this cell.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("../portfolio").resolve()))
from _nport_lookthrough import nport_holdings, NportUnavailable  # noqa: E402

def _hhi(weights):
    return sum(w * w for w in weights)

raw_weights = [p["weight"] for p in basket]
hhi_raw = _hhi(raw_weights)
neff_raw = 1.0 / hhi_raw if hhi_raw > 0 else float("nan")

xray_weights = list(effective.values())
hhi_xray = _hhi(xray_weights)
neff_xray = 1.0 / hhi_xray if hhi_xray > 0 else float("nan")

# Naive sector view — each ETF is its own bucket
NAIVE_SECTOR = {
    "VTI": "Broad US Equity", "VXUS": "Broad Intl Equity",
    "VNQ": "Real Estate ETF", "GLD": "Commodity (Gold)",
    "BND": "Bond Fund", "TLT": "Bond Fund", "SHY": "Bond Fund",
    "TIP": "Bond Fund", "SCHP": "Bond Fund",
    "XLE": "Energy", "XLF": "Financials", "XLV": "Health Care",
    "XLU": "Utilities", "XLB": "Materials", "XLI": "Industrials",
    "DBC": "Commodity (Broad)", "VWO": "Broad EM Equity",
}
naive_by_sector = {}
for p in basket:
    sec = NAIVE_SECTOR.get(p["symbol"], "Unknown")
    naive_by_sector[sec] = naive_by_sector.get(sec, 0.0) + p["weight"]

# Free-tier sector map — GICS-aligned; sourced from public 10-K SIC codes.
# Keys are issuer 'name' as it appears on N-PORT rows (with common variants).
FREE_SECTOR_MAP = {
    # Technology
    "Microsoft Corp.": "Technology", "Microsoft Corp": "Technology",
    "NVIDIA Corp.": "Technology", "NVIDIA Corp": "Technology",
    "Apple Inc.": "Technology", "Apple Inc": "Technology",
    "Broadcom Inc.": "Technology", "Broadcom Inc": "Technology",
    "Advanced Micro Devices, Inc.": "Technology",
    "Cisco Systems, Inc.": "Technology",
    "Applied Materials, Inc.": "Technology",
    "Lam Research Corp.": "Technology", "KLA Corp.": "Technology",
    "Intel Corp.": "Technology", "Texas Instruments Inc.": "Technology",
    "Analog Devices, Inc.": "Technology", "Oracle Corp.": "Technology",
    "Salesforce, Inc.": "Technology", "Adobe Inc.": "Technology",
    "ServiceNow, Inc.": "Technology",
    "Palantir Technologies Inc.": "Technology",
    "Micron Technology, Inc.": "Technology",
    "International Business Machines Corp.": "Technology",
    "QUALCOMM Inc.": "Technology", "Intuit Inc.": "Technology",
    # Communication Services
    "Alphabet Inc.": "Communication Services",
    "Alphabet Inc": "Communication Services",
    "Meta Platforms, Inc.": "Communication Services",
    "Netflix, Inc.": "Communication Services",
    "T-Mobile US, Inc.": "Communication Services",
    "Comcast Corp.": "Communication Services",
    "Verizon Communications Inc.": "Communication Services",
    "AT&T Inc.": "Communication Services",
    # Consumer Cyclical
    "Amazon.com, Inc.": "Consumer Cyclical",
    "Tesla, Inc.": "Consumer Cyclical",
    "Home Depot, Inc.": "Consumer Cyclical",
    "McDonald's Corp.": "Consumer Cyclical",
    "Booking Holdings Inc.": "Consumer Cyclical",
    # Consumer Defensive
    "Walmart Inc.": "Consumer Defensive",
    "Costco Wholesale Corp.": "Consumer Defensive",
    "PepsiCo, Inc.": "Consumer Defensive",
    "The Coca-Cola Co.": "Consumer Defensive",
    "Procter & Gamble Co.": "Consumer Defensive",
    # Financial Services
    "JPMorgan Chase & Co.": "Financial Services",
    "Berkshire Hathaway Inc.": "Financial Services",
    "Visa Inc.": "Financial Services",
    "Mastercard Inc.": "Financial Services",
    "Bank of America Corp.": "Financial Services",
    # Healthcare
    "UnitedHealth Group Inc.": "Healthcare",
    "Eli Lilly and Co.": "Healthcare",
    "Johnson & Johnson": "Healthcare",
    "AbbVie Inc.": "Healthcare", "Merck & Co., Inc.": "Healthcare",
    "Amgen Inc.": "Healthcare", "Gilead Sciences, Inc.": "Healthcare",
    "Intuitive Surgical, Inc.": "Healthcare",
    # Energy
    "Exxon Mobil Corp.": "Energy", "Chevron Corp.": "Energy",
    # Industrials
    "Honeywell International Inc.": "Industrials",
    "Linde PLC": "Basic Materials",
}

def _rollup_free(rows, weight_scale=1.0, top_n=50):
    """Free-tier sector rollup — FREE_SECTOR_MAP + Unknown."""
    out, counts = {}, {}
    for i, r in enumerate(sorted(rows, key=lambda x: -(x.get("weight") or 0.0))):
        w = (r.get("weight") or 0.0) * weight_scale
        if w <= 0:
            continue
        asset_cat = r.get("asset_category") or ""
        if asset_cat in ("DBT", "ABS-APCP", "ABS-CBDO", "ABS-MBS",
                         "ABS-O", "ABS-CDO", "ABS-CMBS", "ABS-RMBS", "LOAN"):
            bucket = "Fixed Income"
        elif asset_cat == "STIV":
            bucket = "Short-Term / Cash"
        elif asset_cat == "RE":
            bucket = "Real Estate"
        elif asset_cat == "COMM":
            bucket = "Commodity"
        elif i < top_n and r.get("name") in FREE_SECTOR_MAP:
            bucket = FREE_SECTOR_MAP[r["name"]]
        else:
            bucket = "Unknown"
        out[bucket] = out.get(bucket, 0.0) + w
        counts[bucket] = counts.get(bucket, 0) + 1
    return out, counts

xray_by_sector = {}
unknown_count = 0
for pos in basket:
    sym = pos["symbol"]; w = pos["weight"]
    if sym in COMMODITY_TRUSTS:
        xray_by_sector["Commodity"] = xray_by_sector.get("Commodity", 0.0) + w
        continue
    if sym in BOND_ETFS:
        xray_by_sector["Fixed Income"] = xray_by_sector.get("Fixed Income", 0.0) + w
        continue
    if sym in EQUITY_ETFS:
        try:
            rows = nport_holdings(sym)
        except NportUnavailable:
            xray_by_sector[sym] = xray_by_sector.get(sym, 0.0) + w
            continue
        sec_w, cnt = _rollup_free(rows, weight_scale=w, top_n=50)
        for k, v in sec_w.items():
            xray_by_sector[k] = xray_by_sector.get(k, 0.0) + v
        unknown_count += cnt.get("Unknown", 0)
    else:
        # single-name equity — Track B has no free ticker->sector API;
        # fall back to Unknown. Variant A has no single-name equities.
        xray_by_sector["Unknown"] = xray_by_sector.get("Unknown", 0.0) + w

print(f"{'Concentration':<20}{'Naive':>12}{'X-Ray':>12}{'Delta':>12}")
print("-" * 56)
print(f"{'HHI':<20}{hhi_raw:>12.4f}{hhi_xray:>12.4f}{hhi_xray-hhi_raw:>+12.4f}")
print(f"{'Effective-N':<20}{neff_raw:>12.2f}{neff_xray:>12.2f}{neff_xray-neff_raw:>+12.2f}")
print()
print("Naive sector view (top 6):")
for sec, w in sorted(naive_by_sector.items(), key=lambda kv: -kv[1])[:6]:
    print(f"  {sec:<24}{w*100:>7.1f}%")
print()
print("Look-through sector view (top 8, via N-PORT + FREE_SECTOR_MAP):")
for sec, w in sorted(xray_by_sector.items(), key=lambda kv: -kv[1])[:8]:
    print(f"  {sec:<24}{w*100:>7.2f}%")
print()
print(f"Rollup transparency: {unknown_count} constituent rows fell into Unknown "
      f"across the equity ETFs (free-tier sector-map gap — see §11).")


## 7. Single-name spot-check — honest hand-off to Track A NB02

Track A NB08 §7 runs the Analysis 7-phase pipeline on MSFT (the largest constituent of VTI, which dominates the look-through view). The 7-phase composite requires `fmp_cached` — there is no free-authoritative substitute for the earnings-estimate / analyst-target / composite-score path. Same honest hand-off NB02 (Track B) §7 already documented: we surface the Track A number the reader should compare against, and leave the composite computation itself to Track A.

**From the last shipped Track A run (2026-07-21):**

- `p7.action_label` — **Avoid**
- `p7.composite_score` — **2.52** (7-phase weighted composite; scale 1-4)
- `p7.entry_quality` — **Wait**

Reproduce under Track A by opening [`../portfolio/08-analyst-recommendations-basket.ipynb`](../portfolio/08-analyst-recommendations-basket.ipynb) §7 or [`../portfolio/02-single-name-deep-dive.ipynb`](../portfolio/02-single-name-deep-dive.ipynb) and running under `.venv_portfolio` with an `fmp_cached` key configured.

The single-name score does not change the basket-level conclusions in §5–§6 above — those are pure arithmetic on N-PORT weights and free-tier sector labels — but it is the number Sam would want on the fridge next to the basket table if the composite were free.


## 8. Backtest — buy-and-hold on the 16-ETF universe vs SPY (provider='cboe')

Same backtest as Track A NB08 §8 — buy-and-hold on the 16-ETF universe over 2023-01-03 → 2024-12-31 vs SPY. Only difference: `provider="cboe"` threaded through the engine so the historical price source is free-authoritative. NB06 (Track B) established the pattern; PR #1455 fixed the CBOE historical path so `openbb_backtest` accepts it cleanly.

The point is not to prove the basket beats SPY — a US-heavy 2023-2024 window flatters SPY heavily. The point is to have honest Sharpe / vol / MaxDD / CAGR numbers under a free provider.

CBOE is price-only (no dividend reinvestment). For dividend-paying sleeves (BND, TLT, VNQ, XLU, VXUS) the total-return figures run ~1-2%/yr lower than a total-return-adjusted Track A run. NB06 (Track B) §0.5 has the fine print. On the shipped 2023-2024 window this delta is small enough that the qualitative reading — mid-single-digit CAGR, sub-10% vol, sub-Sharpe-1 — matches Track A.


In [ ]:
# [Track B / NB08 §8] Backtest — buy_and_hold on 16-ETF universe vs SPY (provider='cboe')
from datetime import date
from decimal import Decimal
from openbb import obb
from openbb_backtest.models import (
    BacktestConfig, CommissionModel, SlippageModel, ComputeConfig,
)
import warnings; warnings.filterwarnings("ignore")

UNIVERSE = [p["symbol"] for p in basket]
START, END = date(2023, 1, 3), date(2024, 12, 31)
PROVIDER = "cboe"

config_a = BacktestConfig(
    strategy="buy_and_hold",
    universe=UNIVERSE,
    start=START,
    end=END,
    initial_cash=Decimal("100000"),
    commission=CommissionModel(kind="flat", value=Decimal("0"), min_per_trade=Decimal("0")),
    slippage=SlippageModel(kind="fixed_bps", value=Decimal("0")),
    compute=ComputeConfig(),
    benchmark="SPY",
    frequency="daily",
)
STRATEGY_PARAMS_A = {"symbols": UNIVERSE}

print(f"Universe:     {len(UNIVERSE)} ETFs (Variant A)")
print(f"Window:       {START} -> {END}")
print(f"Strategy:     {config_a.strategy} (equal-weight)")
print(f"Benchmark:    {config_a.benchmark}")
print(f"Provider:     {PROVIDER}  (free-tier, price-only)")
print()

metrics_a = {}
try:
    result_a = obb.backtest.run(config_a, strategy_params=STRATEGY_PARAMS_A, provider=PROVIDER)
    m = result_a.results.metrics
    for field in ("sharpe", "volatility", "max_drawdown", "cagr", "sortino", "calmar"):
        val = getattr(m, field, None)
        if val is None:
            continue
        metrics_a[field] = float(val)
    print(f"Variant A buy-and-hold summary metrics (provider={PROVIDER}):")
    for field, val in metrics_a.items():
        print(f"  {field:<18}{val:+.4f}")
except Exception as exc:  # noqa: BLE001
    print(f"Backtest failed: {type(exc).__name__}: {str(exc)[:200]}")
    print("Falling back to a narrative note only — the config above shows the intended run.")


## 9. Fortress swap — bring your own reference basket

If you already have a reference basket ("Fortress" was the user's shorthand — could equally be an in-house IPS, a Ric Edelman lineup, or a JP Morgan CIO letter's model portfolio), you can swap it in with a single environment variable. The notebook looks for `FORTRESS_BASKET_JSON` pointing at a file with the same `[{"symbol": ..., "weight": ...}, ...]` shape as §4 and, when present, tells you how to re-run §5–§6 against it.

For the shipped run there is no override — the cell below documents the swap-in path without executing it, so the review numbers stay tied to the §4 basket every reader can reproduce.


In [ ]:
# [Track B / NB08 §9] Optional swap — bring your own reference basket
import os, json
from pathlib import Path

override = os.environ.get("FORTRESS_BASKET_JSON")
if not override:
    print("no override set - using default basket above")
    print("(to swap in your own basket, set FORTRESS_BASKET_JSON=<path-to-json>")
    print(" pointing at a file shaped like .notebook_state/analyst_basket_free.json)")
else:
    path = Path(override)
    if not path.exists():
        print(f"FORTRESS_BASKET_JSON={override} does not exist - keeping default basket")
    else:
        override_rows = json.loads(path.read_text(encoding="utf-8"))
        total = sum(r.get("weight", 0.0) for r in override_rows)
        print(f"Loaded override from {path}")
        print(f"  {len(override_rows)} positions, weights sum to {total*100:.1f}%")
        print("  (re-run §5-§6 with `basket = override_rows` in a scratch cell to")
        print("   review this basket instead of the default)")


## 10. The self-maintained alternative — five ETFs, one rebalance a year

Variant A above is the diversified all-weather book. It has 16 sleeves across five frameworks and a look-through into hundreds of underlying issuers. It is also a book you have to *maintain* — sixteen positions, sixteen expense ratios to watch, sixteen rebalance decisions once a year. That maintenance cost is real. It's the reason so many "sophisticated" baskets underperform a three-fund portfolio kept for a decade with discipline.

**Variant B — Low-cost self-maintained (5 ETFs, annual rebalance).**

| Ticker | Name | Weight | ER | Why |
|---|---|---:|---:|---|
| VTI | Vanguard Total US Stock Market | 55% | 0.03% | Broad US equity spine |
| VXUS | Vanguard Total International Stock | 20% | 0.05% | Broad ex-US |
| BND | Vanguard Total Bond Market | 15% | 0.03% | Investment-grade agg |
| VNQ | Vanguard Real Estate | 5% | 0.12% | REIT diversifier |
| SCHP | Schwab TIPS ETF | 5% | 0.03% | Real-rate inflation hedge |

**Weighted ER math** — (0.55 × 0.03) + (0.20 × 0.05) + (0.15 × 0.03) + (0.05 × 0.12) + (0.05 × 0.03) ≈ **0.038%**. On a $500k book that's roughly **$190/yr in fund fees**. Variant A's weighted ER runs **~0.10-0.16%** on the same mix (≈ **$500-800/yr**). The delta isn't the point; the *reliability* is. You cannot forget to pay a low ER — it just happens.

*The code cell below writes Variant B to `.notebook_state/analyst_basket_low_cost_free.json` (Track B suffix — Track A's `analyst_basket_low_cost.json` is left untouched), prints the weighted-ER math, and runs the same Variant B backtest as Track A under `provider='cboe'` for a head-to-head comparison against Variant A.*


In [ ]:
# [Track B / NB08 §10] Variant B low-cost basket + weighted-ER + head-to-head backtest
import json
from pathlib import Path

BASKET_LOW_COST = [
    {"symbol": "VTI",  "weight": 0.55, "expense_ratio": 0.0003, "role": "US equity spine"},
    {"symbol": "VXUS", "weight": 0.20, "expense_ratio": 0.0005, "role": "Ex-US equity"},
    {"symbol": "BND",  "weight": 0.15, "expense_ratio": 0.0003, "role": "Investment-grade agg"},
    {"symbol": "VNQ",  "weight": 0.05, "expense_ratio": 0.0012, "role": "REIT diversifier"},
    {"symbol": "SCHP", "weight": 0.05, "expense_ratio": 0.0003, "role": "TIPS inflation hedge"},
]

total_wB = sum(p["weight"] for p in BASKET_LOW_COST)
assert abs(total_wB - 1.0) < 1e-9, f"Variant B weights sum to {total_wB}, not 1.0"

state = Path(".notebook_state")
state.mkdir(exist_ok=True)
# Track B path — DO NOT overwrite Track A's analyst_basket_low_cost.json
basket_b_path = state / "analyst_basket_low_cost_free.json"
basket_b_path.write_text(json.dumps(BASKET_LOW_COST, indent=2), encoding="utf-8")

# Weighted ER math
weighted_er_B = sum(p["weight"] * p["expense_ratio"] for p in BASKET_LOW_COST)
# Variant A ER estimate — Vanguard/State-Street/Invesco published ERs.
VARIANT_A_ER = {
    "VTI": 0.0003, "VXUS": 0.0005, "VNQ": 0.0012, "GLD": 0.0040,
    "BND": 0.0003, "TLT": 0.0015, "SHY": 0.0015, "TIP": 0.0019,
    "XLE": 0.0009, "XLF": 0.0009, "XLV": 0.0009, "XLU": 0.0009,
    "XLB": 0.0009, "XLI": 0.0009, "DBC": 0.0085, "VWO": 0.0007,
}
weighted_er_A = sum(p["weight"] * VARIANT_A_ER.get(p["symbol"], 0.0) for p in basket)

print(f"Wrote {basket_b_path}  ({len(BASKET_LOW_COST)} ETFs, weights sum to {total_wB*100:.1f}%)")
print("(Track A's .notebook_state/analyst_basket_low_cost.json NOT modified by this notebook.)")
print()
print(f"{'Ticker':<6}{'Weight':>8}{'ER':>10}   Role")
print("-" * 64)
for p in BASKET_LOW_COST:
    print(f"{p['symbol']:<6}{p['weight']*100:>7.1f}%{p['expense_ratio']*100:>9.3f}%   {p['role']}")
print()
print(f"Weighted ER  Variant A (16 ETFs): {weighted_er_A*100:.3f}%  ~= ${weighted_er_A*500_000:,.0f}/yr on $500k")
print(f"Weighted ER  Variant B (5 ETFs):  {weighted_er_B*100:.3f}%  ~= ${weighted_er_B*500_000:,.0f}/yr on $500k")
print(f"Annual-fee delta on $500k:        ${(weighted_er_A - weighted_er_B)*500_000:,.0f}/yr (A - B)")
print()

# Head-to-head backtest — same window, provider='cboe'
from datetime import date
from decimal import Decimal
from openbb import obb
from openbb_backtest.models import (
    BacktestConfig, CommissionModel, SlippageModel, ComputeConfig,
)
import warnings; warnings.filterwarnings("ignore")

UNIVERSE_B = [p["symbol"] for p in BASKET_LOW_COST]
config_b = BacktestConfig(
    strategy="buy_and_hold",
    universe=UNIVERSE_B,
    start=date(2023, 1, 3),
    end=date(2024, 12, 31),
    initial_cash=Decimal("100000"),
    commission=CommissionModel(kind="flat", value=Decimal("0"), min_per_trade=Decimal("0")),
    slippage=SlippageModel(kind="fixed_bps", value=Decimal("0")),
    compute=ComputeConfig(),
    benchmark="SPY",
    frequency="daily",
)
STRATEGY_PARAMS_B = {"symbols": UNIVERSE_B}

metrics_b = {}
try:
    result_b = obb.backtest.run(config_b, strategy_params=STRATEGY_PARAMS_B, provider="cboe")
    m = result_b.results.metrics
    for field in ("sharpe", "volatility", "max_drawdown", "cagr", "sortino", "calmar"):
        val = getattr(m, field, None)
        if val is None:
            continue
        metrics_b[field] = float(val)
except Exception as exc:  # noqa: BLE001
    print(f"Variant B backtest failed: {type(exc).__name__}: {str(exc)[:200]}")

print("Head-to-head backtest (provider='cboe', 2023-01-03 -> 2024-12-31, vs SPY):")
print(f"{'Metric':<18}{'Variant A':>12}{'Variant B':>12}{'B - A':>12}")
print("-" * 54)
for field in ("sharpe", "volatility", "max_drawdown", "cagr", "sortino", "calmar"):
    a_v = metrics_a.get(field)
    b_v = metrics_b.get(field)
    if a_v is None or b_v is None:
        continue
    print(f"  {field:<16}{a_v:>+12.4f}{b_v:>+12.4f}{b_v-a_v:>+12.4f}")
print()
print("Reading: A has more sleeves and usually wins on raw Sharpe in a US-heavy bull")
print("window; B has ~1/3 the weighted ER and one rebalance a year. Reader chooses on")
print("maintenance appetite, not on the two-year Sharpe delta.")


## 11. What is NOT in this notebook (free-tier delta)

Same gaps as Track A NB08 §11 plus the free-tier deltas:

- **Live scraping of TipRanks / Zacks / Morningstar / Seeking Alpha   / ETF.com.** Each has a distinct ToS and rate-limit posture.
- **Analyst grades / consensus price targets** — **no free-  authoritative source**. yfinance's `recommendations` field is a   scraped best-effort feed with no SLA. Track A pulls this via   `fmp_cached`; Track B honestly documents the gap and moves on.   If you need it, click through to the aggregator sites in §3 or   add an `fmp_cached` key.
- **Analysis 7-phase composite on MSFT (§7).** Requires   `fmp_cached`. Track B hands off to Track A NB02 / NB08 §7 with   the last shipped numbers reproduced in prose.
- **Fake-classified sector tail.** ~15-25% of x-ray weight lands in   `Unknown` — the tail below the top-60 mega-caps that our free   sector map doesn't cover. Track A routes this tail through   `fmp_cached` sector profiles. Track B names the gap rather than   fake-classifying via yfinance-scraped labels (#1425).
- **Total-return-adjusted backtest metrics.** CBOE is price-only.   Sharpe / CAGR run ~1-2%/yr lower than Track A on dividend-paying   sleeves. NB06 (Track B) §0.5 has the fine print.
- **Weights-target rebalancing backtest.** Discussed in Track A §8;   Track B inherits the same limitation.
- **Trend-following overlay** (Faber's 10-month MA). Cited in §2 as   the classic Ivy exit rule; not implemented.
- **A forward-return forecast.** The basket has no expected-return   model attached, deliberately.


## 12. 📚 Further reading

Every Investopedia link cited in Track A NB08 (risk parity, all-weather, three-fund, target-date, trend following, Zacks Rank, Morningstar star rating, SEC EDGAR, commodity ETF, efficient frontier, sector rotation, analyst price target, rebalancing, CAGR, Sharpe ratio, fundamental analysis, expense ratio) applies unchanged. Not re-cited here.

**Portfolio construction frameworks (methodology):**

- Ray Dalio — All Weather portfolio (Bridgewater research library): https://www.bridgewater.com/research-library
- John Bogle — Three-fund portfolio (Bogleheads wiki): https://www.bogleheads.org/wiki/Three-fund_portfolio
- Fidelity — Sector rotation framework: https://www.fidelity.com/learning-center/investment-products/mutual-funds/sector-rotation-strategy
- Vanguard — Target-retirement funds: https://investor.vanguard.com/investment-products/list/target-retirement
- Meb Faber — *The Ivy Portfolio* (Cambria): https://mebfaber.com/

**Analyst / signal aggregation destinations** (same six as Track A §3): TipRanks, Zacks, Morningstar, Seeking Alpha, ETF.com, SEC EDGAR.

**Free-authoritative sources used in this notebook:**

- **SEC EDGAR N-PORT** — quarterly holdings for '40-Act US-  registered funds (§5, §6).
- **CBOE EOD** — daily closes for the backtest (§8, §10) via   `provider='cboe'`.

**Books** (unchanged from Track A):

- Jack Bogle — *The Little Book of Common Sense Investing* (Wiley).
- Rick Ferri — *All About Asset Allocation* (McGraw-Hill).

**Series pointers:**

- Track A counterpart: [`../portfolio/08-analyst-recommendations-basket.ipynb`](../portfolio/08-analyst-recommendations-basket.ipynb)
- Series README: [`../portfolio/README.md`](../portfolio/README.md)
- STORY_BIBLE: [`../portfolio/STORY_BIBLE.md`](../portfolio/STORY_BIBLE.md)
